# End-to-End Test Notebook

This notebook runs a compact smoke test over the components implemented so far: dataframe loading, imputers, encoders, scalers, models, and parameter optimization.

In [1]:
import numpy as np

from dataframes.DataFrame import DataFrame
from preprocessing.impute.simple import SimpleImputer, MedianImputer, ModeImputer
from preprocessing.impute.knn import KNNImputer
from preprocessing.encode.label import LabelEncoder
from preprocessing.encode.onehot import OneHotEncoder
from preprocessing.scale.standard import StandardScaler
from preprocessing.scale.minmax import MinMaxScaler
from models.linearregression.linear import LinearRegression
from models.logisticregression.logistic_regression import LogisticRegression
from models.knn.classifier import KNNClassifier
from models.knn.regressor import KNNRegressor
from models.naive.naive import NaiveBayes
from optimization.gridsearch.gridsearch import GridSearch
from optimization.randomsearch.randomsearch import RandomSearch
from optimization.kfold.kfold import KFoldGridSearch
from evaluation.classification.classification import accuracy, classification_report
from evaluation.regression.regression import regression_report

df = DataFrame.load_csv("data.csv")
df

DataFrame([(24.0, 1.0, 29.99, 0.0, nan, 'Basic', 'Mobile', 'on_time', 0.0, 180.0),
           (27.0, 3.0, 49.5, 1.0, 'Rabat', 'Standard', 'Desktop', 'on_time', 0.0, 420.0),
           (31.0, 6.0, 79.0, 2.0, 'Marrakesh', 'Premium', 'Tablet', 'late', 0.0, 860.0),
           (45.0, 15.0, 120.0, 5.0, 'Casablanca', 'Premium', 'Desktop', 'late', 1.0, 1500.0),
           (52.0, 22.0, 140.0, 6.0, 'Tangier', 'Premium', 'Mobile', 'defaulted', 1.0, 1625.0),
           (nan, 4.0, 58.0, 1.0, 'Rabat', 'Standard', 'Mobile', 'on_time', 0.0, 510.0),
           (29.0, nan, 65.0, 2.0, 'Agadir', 'Standard', 'Desktop', 'late', 0.0, 640.0),
           (38.0, 10.0, nan, 3.0, 'Casablanca', 'Premium', 'Tablet', 'late', 1.0, 980.0),
           (41.0, 12.0, 95.0, nan, 'Fes', 'Premium', 'Desktop', 'defaulted', 1.0, 1100.0),
           (35.0, 8.0, 72.0, 2.0, nan, 'Standard', 'Mobile', 'on_time', 0.0, 760.0),
           (33.0, 7.0, 68.0, 1.0, 'Rabat', nan, 'Desktop', 'on_time', 0.0, 710.0),
           (48.0, 18.0, 

## Preprocessing checks

This first block verifies that the imputers, encoders, and scalers can be fitted and transformed on the current sample dataset.

### Test Simple Median Imputer

In [2]:
age_values = np.array(df["age"], dtype=float)
age_imputer = SimpleImputer(strategy=MedianImputer())
age_clean = age_imputer.fit_transform(age_values)


### Test Simple Mode Imputer

In [3]:
payment_values = np.array(df["payment_status"], dtype=object)
payment_imputer = SimpleImputer(strategy=ModeImputer())
payment_clean = payment_imputer.fit_transform(payment_values)

### Test KNN Imputer

In [4]:
knn_imputed_df = KNNImputer(n_neighbors=3).fit_transform(df.copy())
df = knn_imputed_df

### Test OneHot Encoder

In [5]:
city_values = np.array(df["city"], dtype=object)
city_encoder = LabelEncoder(handle_unknown="ignore")
city_encoded = city_encoder.fit_transform(city_values)

### Test OneHot Encoder

In [6]:
plan_values = np.array(df["plan"], dtype=object)
plan_encoder = OneHotEncoder(handle_unknown="ignore")
plan_onehot = plan_encoder.fit_transform(plan_values)

### Test Standard Scaler

In [7]:
monthly_spend = np.array(df["monthly_spend"], dtype=float)
standard_scaler = StandardScaler()
monthly_spend_standard = standard_scaler.fit_transform(monthly_spend)

### Test MinMax Scaler

In [8]:
minmax_scaler = MinMaxScaler()
monthly_spend_minmax = minmax_scaler.fit_transform(monthly_spend)

### Test Summary

In [9]:
preprocessing_summary = {
    "features": df.get_features(),
    "median_age_fill": float(age_imputer.value),
    "mode_payment_fill": payment_imputer.value,
    "knn_missing_age_after": int(np.isnan(np.array(knn_imputed_df["age"], dtype=float)).sum()),
    "city_classes": city_encoder.classes_,
    "plan_onehot_shape": tuple(plan_onehot.shape),
    "standard_mean_approx": float(np.round(np.mean(monthly_spend_standard), 6)),
    "minmax_range": (
        float(np.round(np.min(monthly_spend_minmax), 6)),
        float(np.round(np.max(monthly_spend_minmax), 6)),
    ),
}

preprocessing_summary

NameError: name 'knn_imputed_df' is not defined

## Shared feature pipeline

This block builds a numeric feature matrix that can be reused by the models and by the optimization utilities.

In [ ]:
device_values = np.array(df["device"], dtype=object)
payment_values = np.array(payment_clean, dtype=object)

device_encoder = LabelEncoder(handle_unknown="ignore")
payment_encoder = LabelEncoder(handle_unknown="ignore")

device_encoded = device_encoder.fit_transform(device_values)
payment_encoded = payment_encoder.fit_transform(payment_values)

base_numeric = np.column_stack([
    age_clean,
    np.array(df["years_experience"], dtype=float),
    np.array(df["monthly_spend"], dtype=float),
    np.array(df["support_tickets"], dtype=float),
    city_encoded.astype(float),
    device_encoded.astype(float),
    payment_encoded.astype(float),
])

scaled_columns = []
for col_idx in range(base_numeric.shape[1]):
    scaler = StandardScaler()
    scaled_columns.append(scaler.fit_transform(base_numeric[:, col_idx]))

X = np.column_stack(scaled_columns + [plan_onehot.astype(float)])
y_classification = np.array(df["churned"], dtype=int)
y_regression = np.array(df["lifetime_value"], dtype=float)

split_idx = 18
X_train, X_test = X[:split_idx], X[split_idx:]
y_cls_train, y_cls_test = y_classification[:split_idx], y_classification[split_idx:]
y_reg_train, y_reg_test = y_regression[:split_idx], y_regression[split_idx:]

{
    "X_shape": tuple(X.shape),
    "classification_train_size": int(len(y_cls_train)),
    "classification_test_size": int(len(y_cls_test)),
}

{'X_shape': (25, 10),
 'classification_train_size': 18,
 'classification_test_size': 7}

## Model smoke tests

The next cells fit the currently usable models and report simple holdout metrics.

In [ ]:
classification_models = {
    "LogisticRegression": LogisticRegression(lr=0.1, max_iters=50),
    "LinearRegression": LinearRegression(lr=0.01),
    "KNNClassifier": KNNClassifier(k=12),
    "NaiveBayes": NaiveBayes(),
}

classification_results = {}

for model_name, model in classification_models.items():
    model.fit(X_train, y_cls_train)
    predictions = model.predict(X_test)
    classification_results[model_name] = {
        "accuracy": round(accuracy(y_cls_test, predictions), 4),
        "report": classification_report(y_cls_test, predictions),
    }
 
classification_results["LogisticRegression" ]

{'accuracy': 1.0,
 'report': {'classes': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1': 1.0,
    'support': 4},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 3}},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'weighted avg': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}}}

In [ ]:
regression_models = {
    "LinearRegression": LinearRegression(lr=0.01, epochs=4000, tol=1e-10),
    "KNNRegressor": KNNRegressor(k=3),
}

regression_results = {}

for model_name, model in regression_models.items():
    model.fit(X_train, y_reg_train)
    predictions = model.predict(X_test)
    regression_results[model_name] = {
        metric_name: round(metric_value, 4)
        for metric_name, metric_value in regression_report(y_reg_test, predictions).items()
    }

regression_results

{'LinearRegression': {'MAE': 46.9247,
  'MSE': 3869.1025,
  'RMSE': 62.2021,
  'R2': 0.9864},
 'KNNRegressor': {'MAE': 126.9048,
  'MSE': 22957.5397,
  'RMSE': 151.5175,
  'R2': 0.9195}}

## Parameter optimization checks

These searches confirm that the optimization utilities can evaluate parameter combinations and return the best configuration.

In [ ]:
grid_search = GridSearch(
    LogisticRegression,
    param_grid={"lr": [0.05, 0.1, 0.2], "max_iters": [1000, 3000]},
    metric=accuracy,
    greater_is_better=True,
)

random_search = RandomSearch(
    KNNClassifier,
    param_distributions={"k": [1, 3, 5, 7]},
    metric=accuracy,
    n_iter=4,
    greater_is_better=True,
    seed=42,
)

kfold_grid_search = KFoldGridSearch(
    LogisticRegression,
    param_grid={"lr": [0.05, 0.1], "max_iters": [1000, 3000]},
    metric=accuracy,
    n_splits=3,
    shuffle=True,
    seed=42,
)

_, best_grid_params, best_grid_score = grid_search.search(X_train, y_cls_train, X_test, y_cls_test)
_, best_random_params, best_random_score = random_search.search(X_train, y_cls_train, X_test, y_cls_test)
_, best_kfold_params, best_kfold_score = kfold_grid_search.search(X, y_classification)

{
    "grid_search": {
        "best_params": best_grid_params,
        "best_score": round(best_grid_score, 4),
    },
    "random_search": {
        "best_params": best_random_params,
        "best_score": round(best_random_score, 4),
    },
    "kfold_grid_search": {
        "best_params": best_kfold_params,
        "best_score": round(best_kfold_score, 4),
    },
}

{'grid_search': {'best_params': {'lr': 0.05, 'max_iters': 1000},
  'best_score': 1.0},
 'random_search': {'best_params': {'k': np.int64(1)}, 'best_score': 0.8571},
 'kfold_grid_search': {'best_params': {'lr': 0.05, 'max_iters': 1000},
  'best_score': 0.9213}}

## Note

This notebook focuses on the modules that are already usable at runtime. If a new model is completed later, it can be plugged into the shared feature pipeline and evaluated with the same pattern.